# Quickstart: round-trip a golden codec

Load a published **compressionKIT** golden codec, compress and decompress a
physiological frame, and measure the true compression ratio and fidelity.

**No dataset required.** Every deploy package ships a small set of
representative, license-safe reference frames (`reference_vectors.npz`) that we
use here, so the notebook runs anywhere.

> Loading from HuggingFace needs internet access. To run fully offline, point
> `CODEC_SOURCE` at a local deploy package you built with
> `compressionkit golden run ...`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from compressionkit.evaluation.metrics import compute_signal_metrics
from compressionkit.runtime import load_codec, resolve_deploy_dir

# A published golden codec (downloads from HuggingFace) ...
CODEC_SOURCE = "Ambiq/compressionkit-ppg-4x"
# ... or a local deploy package you built yourself:
# CODEC_SOURCE = "results/ppg_rvq_64hz_04x_golden/deploy"

In [ ]:
codec = load_codec(CODEC_SOURCE)

print(f"name        : {codec.name}")
print(f"modality    : {codec.modality}")
print(f"sample_rate : {codec.sample_rate} Hz")
print(f"frame_size  : {codec.frame_size} samples ({codec.frame_size / codec.sample_rate:.2f} s)")
print(f"target CR   : {codec.target_cr:g}x")

## Representative frames

`reference_vectors.npz` bundles in-distribution input frames together with the
exact reconstructions the reference runtime should produce — handy for both
demos and validation.

In [ ]:
deploy = Path(resolve_deploy_dir(CODEC_SOURCE))
ref = np.load(deploy / "reference_vectors.npz")

frames = ref["input_frames"][:, 0, :, 0].astype("float32")  # (N, frame_size)
print(f"Loaded {len(frames)} reference frames of {frames.shape[1]} samples")

## Round-trip a single frame

Compression ratio is reported against a **32-bit float** sample baseline — the
toolkit's reference raw representation, which matches the headline `target CR`.

In [ ]:
frame = frames[0]
encoded = codec.compress(frame)
recon = codec.decompress(encoded)

raw_bits = frame.size * 32  # 32-bit float baseline
true_cr = raw_bits / encoded.nbits
m = compute_signal_metrics(frame, recon)

print(f"encoded bits : {encoded.nbits}  ->  measured CR {true_cr:.2f}x (vs 32-bit float)")
print(f"PRD          : {m['prd_percent']:.2f}%")
print(f"cosine sim   : {m['cosine_similarity']:.4f}")

In [ ]:
t = np.arange(frame.size) / codec.sample_rate
plt.figure(figsize=(10, 3))
plt.plot(t, frame, label="original", lw=1.5)
plt.plot(t, recon, label="reconstruction", lw=1.2, alpha=0.85)
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title(f"{codec.name}  ·  PRD {m['prd_percent']:.2f}%  ·  CR {true_cr:.2f}x")
plt.legend()
plt.tight_layout()
plt.show()

## Aggregate over many frames

A single number hides the spread. Round-trip a batch and look at the
distribution of fidelity alongside the realized compression ratio.

In [ ]:
N = min(500, len(frames))
batch = frames[:N]

total_bits = 0
prd, cos = [], []
for f in batch:
    e = codec.compress(f)
    r = codec.decompress(e)
    total_bits += e.nbits
    mm = compute_signal_metrics(f, r)
    prd.append(mm["prd_percent"])
    cos.append(mm["cosine_similarity"])

prd = np.asarray(prd)
cos = np.asarray(cos)
print(f"frames evaluated : {N}")
print(f"measured CR      : {(batch.size * 32) / total_bits:.2f}x")
print(f"PRD %            : median {np.median(prd):.2f}   mean {prd.mean():.2f}")
print(f"cosine           : median {np.median(cos):.4f}   mean {cos.mean():.4f}")

In [ ]:
plt.figure(figsize=(10, 3))
plt.hist(prd, bins=30, color="#4C78A8")
plt.axvline(np.median(prd), color="k", ls="--", label=f"median {np.median(prd):.2f}%")
plt.xlabel("PRD %")
plt.ylabel("frames")
plt.title("Per-frame fidelity distribution")
plt.legend()
plt.tight_layout()
plt.show()

## Next steps

- Swap `CODEC_SOURCE` for any tier in the [Model Zoo](https://ambiqai.github.io/compressionkit/models/) (e.g. `Ambiq/compressionkit-ecg-8x`).
- Evaluate on **your own recordings** — see `02_evaluate_on_your_data.ipynb`.
- Reproduce a golden end-to-end: `compressionkit golden run ppg-rvq-4x`.